In [ ]:
import uuid, time
from src.spark.spark import get_spark_session
from src.utils.logging_config import setup_logging
from src.io.writer import DataFrameWriter

In [2]:
layer = "gold"
dataset = "dim_dates"

# Load Spark Session
spark = get_spark_session()

# Load logging
log = setup_logging(f"{dataset}_{layer}")

# Create logical variables
run_id = str(uuid.uuid4())

In [5]:
log.info(f"[PARAMS] dataset={dataset} layer={layer}")

start = time.time()
log.info(f"[GOLD][START] run_id={run_id}")

try:

    writer = DataFrameWriter(spark)
    
    dim_dates = spark.sql("""
        SELECT 
            CAST(date_col AS date) AS date_id,
            YEAR(date_col) AS year,
            MONTH(date_col) AS month,
            DAY(date_col) AS day,
            DAYOFWEEK(date_col) AS day_of_week
        FROM (
            SELECT EXPLODE(SEQUENCE(
                '2016-01-01'::timestamp,
                CAST(current_date() AS timestamp) + INTERVAL 90 DAY,
                INTERVAL 1 DAY
            )) AS date_col
        )
        ORDER BY date_col
    """)
    
    writer.write_delta_batch(dim_dates, "olist", "gold", "dim_dates", mode="overwrite")
    duration_s = round(time.time() - start, 3)
    log.info(f"[GOLD][SUCCESS] run_id={run_id} dataset={dataset} duration_s={duration_s}s")

except Exception as e:
    duration_s = round(time.time() - start, 3)
    log.error(f"[GOLD][FAILED] run_id={run_id} duration_s={duration_s} error={repr(e)}")
    raise